# 15 — Full FE + best blend

**Goal:** blend LightGBM and XGBoost probabilities on the full engineered feature set.

## Why this experiment?
A simple weighted blend is often close to stacking, with less complexity.
This is the maximum-performance candidate for the leaderboard.

## Approach
1. Full pricing + time + geo features.
2. Train a strong LightGBM and a strong XGBoost.
3. Blend probabilities: 0.60 LGBM + 0.40 XGB.
4. Evaluate once on the shared test set.

## What changed?
- Ensemble of two boosted models (fixed weights)
- Full FE included
- Still compare against experiment 14 before deploying

## Features used in this notebook
- Full pricing + time + geo features
- **How selected:** reuse the strongest FE; blend LGBM + XGB probabilities
- **Why:** near-top ROC-AUC with a simple ensemble


### Setup
Shared data load and fixed split.


In [2]:
import os
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "shared" / "protocol.py").exists():
        EXPERIMENT_ROOT = candidate
        break
    if (candidate / "hyperack_exp" / "shared" / "protocol.py").exists():
        EXPERIMENT_ROOT = candidate / "hyperack_exp"
        break
else:
    raise RuntimeError("Run this notebook from the HyperAck project directory.")
os.chdir(EXPERIMENT_ROOT)
sys.path.insert(0, str(EXPERIMENT_ROOT))

from shared.protocol import (
    add_geo_features,
    add_pricing_features,
    add_time_features,
    base_features,
    evaluate,
    load_clean_df,
    make_xy,
    save_result,
    split_frame,
    actual_vs_predicted_report,
)

RANDOM_STATE = 42
train_df, test_df = split_frame(load_clean_df())


### Build blend
Define LGBM + XGB and the probability blend.


In [3]:
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

X_train, y_train = make_xy(train_df, pricing=True, time_features=True, geo=True)
X_test, y_test = make_xy(test_df, pricing=True, time_features=True, geo=True)

class ProbabilityBlend:
    def __init__(self):
        self.models = [
            Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", LGBMClassifier(n_estimators=1000, learning_rate=0.03, num_leaves=47, min_child_samples=25, reg_lambda=1.5, random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1))]),
            Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", XGBClassifier(n_estimators=1000, max_depth=6, learning_rate=0.03, subsample=0.9, colsample_bytree=0.9, min_child_weight=3, reg_lambda=2.0, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1))]),
        ]
        self.weights = np.array([0.60, 0.40])

    def fit(self, X, y):
        for candidate in self.models:
            candidate.fit(X, y)
        return self

    def predict_proba(self, X):
        positive = sum(weight * candidate.predict_proba(X)[:, 1] for weight, candidate in zip(self.weights, self.models))
        return np.column_stack([1 - positive, positive])

model = ProbabilityBlend()


### Features selected and why

Same full engineered matrix as notebooks 08/09/15’s peers.  
No new columns — the experiment is about **blending two strong models** on those features.

Next cell lists every selected column.


In [4]:
feature_why = {
    "deliverey_category_id": "Delivery type — some categories get accepted more often",
    "weekday": "Day of week — weekday vs weekend courier behavior",
    "time_bucket": "Coarse time-of-day bucket from the raw data",
    "total_distance": "Trip length — longer trips can be harder to accept",
    "sum_product": "Order size / number of products",
    "source_latitude": "Pickup latitude — area effects",
    "source_longitude": "Pickup longitude — area effects",
    "destination_latitude": "Drop-off latitude — area effects",
    "destination_longitude": "Drop-off longitude — area effects",
    "first_customer_fare": "First offered customer price (usually known early)",
    "final_customer_fare": "Final customer price — strong but may be post-decision",
    "final_biker_fare": "Final courier pay — strong but may be post-decision",
    "geo_cluster": "Train-only KMeans region of the trip (pickup+drop-off)",
    "log_distance": "Log distance — softens very long trips",
    "first_fare_per_km": "First fare ÷ distance — pay vs effort",
    "final_customer_fare_per_km": "Final customer fare ÷ distance",
    "customer_fare_delta": "Final − first customer fare (price change)",
    "customer_fare_change_pct": "Relative fare change vs first offer",
    "biker_customer_gap": "Biker fare − customer fare (split / margin)",
    "biker_fare_per_km": "Courier pay per km",
    "log_final_customer_fare": "Log of final customer fare",
    "log_final_biker_fare": "Log of final biker fare",
    "hour": "Exact hour of order creation",
    "is_rush_hour": "Lunch/evening peak flag",
    "is_weekend": "Weekend flag",
    "hour_sin": "Cyclical hour (sin) so 23 is near 0",
    "hour_cos": "Cyclical hour (cos)",
    "weekday_sin": "Cyclical weekday (sin)",
    "weekday_cos": "Cyclical weekday (cos)",
    "day_of_month": "Calendar day — mild monthly pattern",
    "haversine_km": "Great-circle route distance in km",
    "latitude_delta": "North/south trip span",
    "longitude_delta": "East/west trip span",
    "geo_bearing_sin": "Trip direction (sin of bearing)",
    "geo_bearing_cos": "Trip direction (cos of bearing)",
    "distance_x_first_fare": "Interaction: long trip × price",
    "category_x_hour": "Interaction: category × hour",
    "total_distance_qbin": "Train-fitted distance quantile bin",
    "first_customer_fare_qbin": "Train-fitted first-fare quantile bin"
}

cols = list(X_train.columns)
rows = []
for c in cols:
    rows.append({
        "feature": c,
        "why_selected": feature_why.get(c, "Part of this experiment's engineered feature set"),
    })
feature_table = pd.DataFrame(rows)
print(f"Total features selected: {len(cols)}")
print("Columns:")
print(", ".join(cols))
feature_table


Total features selected: 34
Columns:
deliverey_category_id, weekday, time_bucket, total_distance, sum_product, source_latitude, source_longitude, destination_latitude, destination_longitude, first_customer_fare, final_customer_fare, final_biker_fare, hour, is_rush_hour, is_weekend, hour_sin, hour_cos, weekday_sin, weekday_cos, day_of_month, log_distance, first_fare_per_km, final_customer_fare_per_km, customer_fare_delta, customer_fare_change_pct, log_final_customer_fare, biker_customer_gap, biker_fare_per_km, log_final_biker_fare, haversine_km, latitude_delta, longitude_delta, geo_bearing_sin, geo_bearing_cos


,feature,why_selected
0,deliverey_category_id,Delivery type — some categories get accepted m...
1,weekday,Day of week — weekday vs weekend courier behavior
2,time_bucket,Coarse time-of-day bucket from the raw data
3,total_distance,Trip length — longer trips can be harder to ac...
4,sum_product,Order size / number of products
5,source_latitude,Pickup latitude — area effects
6,source_longitude,Pickup longitude — area effects
7,destination_latitude,Drop-off latitude — area effects
8,destination_longitude,Drop-off longitude — area effects
9,first_customer_fare,First offered customer price (usually known ea...


### Evaluate
Held-out metrics for the blend.


In [5]:
metrics = evaluate(model, X_train, y_train, X_test, y_test)
metrics


{'accuracy': 0.9351935193519352,
 'roc_auc': 0.9778997747747747,
 'avg_precision': 0.9775714489158028,
 'f1': 0.9291338582677166,
 'recall': 0.9094412331406551,
 'precision': 0.9496981891348089,
 'threshold': 0.5,
 'fit_seconds': 10.366,
 'predict_seconds': 0.055,
 'confusion_matrix': [[1134, 50], [94, 944]],
 'y_true': array([0, 1, 0, ..., 1, 1, 1], shape=(2222,)),
 'y_pred': array([0, 1, 0, ..., 1, 1, 1], shape=(2222,)),
 'y_prob': array([0.0165198 , 0.99599893, 0.00256704, ..., 0.99809089, 0.99877737,
        0.99911002], shape=(2222,))}

### Actual vs predicted (test set)

After training, we score the **held-out test set** and compare:

1. **Actual** labels (`hyper_ack`) vs **predicted** labels  
2. Confusion matrix (rows = actual, columns = predicted)  
3. Per-class precision / recall / F1  
4. A sample of correct and incorrect rows with predicted probability  

This is only test-set performance — not training rows.


In [ ]:
from IPython.display import display
from shared.protocol import actual_vs_predicted_report

avp = actual_vs_predicted_report(
    metrics["y_true"],
    metrics["y_pred"],
    metrics["y_prob"],
    sample_size=25,
)
print("1) Actual vs predicted class counts")
display(avp["class_counts"])
print("2) Confusion matrix")
display(avp["confusion_matrix"])
print("3) Outcome breakdown")
display(avp["outcomes"])
print("4) Per-class metrics")
display(avp["per_class_metrics"])
print("5) Sample of actual vs predicted rows")
display(avp["prediction_sample"])


### Save result
Write experiment `15`.


In [ ]:
result_path = save_result(
    "15",
    "full_fe_best_blend",
    "0.60 LightGBM + 0.40 XGBoost blend on pricing/time/geographic features",
    metrics,
    best_model="ProbabilityBlend(LGBMClassifier, XGBClassifier)",
    notes="Fixed blend is a robust final candidate; compare against stack and standalone tuned models.",
    feature_count=X_train.shape[1],
)
pd.Series(metrics).drop("confusion_matrix").sort_index(), result_path


## What to look at
- Is this the top ROC-AUC on the leaderboard?
- If yes, re-check the leakage-safe score (14) for real-world use.
